# How small can you go?

Every aggregation trades **size** for **accuracy**. You have two levers — the
number of typical **periods** and the number of **segments** per period (see
[Segmentation](segmentation.ipynb)) — and this page is about choosing them: first
by hand, then by letting tsam search the whole space for you.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The trade-off, by hand

More time steps means more accuracy but less compression. Sweeping the number of
periods shows the classic diminishing-returns curve — error drops fast, then
flattens. The "knee" is where extra periods stop paying off:

In [ ]:
rows = []
for k in [2, 4, 6, 8, 12, 16, 24]:
    r = tsam.aggregate(data, n_clusters=k, period_duration="1D")
    rows.append(
        {
            "n_clusters": k,
            "timesteps": k * r.n_timesteps_per_period,
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
sweep = pd.DataFrame(rows)
px.line(
    sweep,
    x="timesteps",
    y="rmse",
    markers=True,
    title="Accuracy vs. size — number of periods",
)

## Why two levers, not one

Suppose you fix the budget at **48 time steps** (`n_clusters x n_segments`). You
could keep just **2 days at full 24-hour resolution**, or trade detail within the
day for **more days at coarser resolution** — 12 days of 4 segments each, say.
Same budget, very different accuracy:

In [ ]:
from tsam import SegmentConfig

budget = 48
splits = [(2, 24), (4, 12), (6, 8), (8, 6), (12, 4), (16, 3), (24, 2)]
rows = []
for n_clusters, n_segments in splits:
    r = tsam.aggregate(
        data,
        n_clusters=n_clusters,
        period_duration="1D",
        segments=SegmentConfig(n_segments=n_segments),
    )
    rows.append(
        {
            "split": f"{n_clusters}d x {n_segments}seg",
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
split = pd.DataFrame(rows)
px.bar(
    split,
    x="split",
    y="rmse",
    title=f"Same {budget}-step budget, different day/segment splits",
)

Both extremes lose: a handful of highly-detailed days can't represent the variety
of the six weeks, and many ultra-coarse days throw away the daily shape. The sweet
spot sits in the middle — and it moves with the budget, which is exactly the chore
the search below removes.

## Both levers at once: a target reduction

Periods and segments interact, so searching by hand gets tedious fast. Give
[`find_optimal_combination`](../api/tuning.md) a target **data reduction** and it
searches period/segment combinations, returning the most accurate one that hits
your budget:

In [ ]:
from tsam.tuning import find_optimal_combination

best = find_optimal_combination(
    data,
    data_reduction=0.1,
    period_duration="1D",  # keep ~10% of the time steps
    n_jobs=1,
    show_progress=False,
)
print(f"best within budget: {best.n_clusters} periods x {best.n_segments} segments")
print(f"  time steps: {best.n_clusters * best.n_segments}  |  RMSE: {best.rmse:.4f}")

## Map the whole frontier

To choose deliberately instead of against one fixed budget,
[`find_pareto_front`](../api/tuning.md) finds the best aggregation at each size —
the accuracy-vs-size frontier:

In [ ]:
from tsam.tuning import find_pareto_front

pareto = find_pareto_front(
    data,
    period_duration="1D",
    timesteps=range(12, 96, 12),
    n_jobs=1,
    show_progress=False,
)
pareto.summary

In [ ]:
pareto.plot()

## What the search decides

The frontier isn't just *how accurate* at each size — it's also *how* tsam spends
the budget. Plotting the chosen periods and segments side by side shows the
pattern: it adds periods first, then reaches for an extra segment as the budget
grows.

In [ ]:
mix = pareto.summary.melt(
    id_vars="timesteps",
    value_vars=["n_clusters", "n_segments"],
    var_name="lever",
    value_name="count",
)
mix["lever"] = mix["lever"].map({"n_clusters": "periods", "n_segments": "segments"})
px.bar(
    mix,
    x="timesteps",
    y="count",
    color="lever",
    barmode="group",
    title="How the search splits each budget",
)

## Pull a result off the frontier

The frontier is not just a picture: look up the best aggregation for a given size
— or for a given accuracy — and use it directly. `find_by_timesteps` and
`find_by_rmse` both return an ordinary `AggregationResult`:

In [ ]:
small = pareto.find_by_timesteps(36)  # tightest budget on the curve
print(
    f"at 36 steps: {small.n_clusters} periods x {small.n_segments} segments, "
    f"RMSE {small.accuracy.rmse.mean():.4f}"
)

accurate = pareto.find_by_rmse(0.10)  # smallest aggregation under an RMSE target
print(
    f"under RMSE 0.10: {accurate.n_clusters} periods x {accurate.n_segments} segments, "
    f"{accurate.n_clusters * accurate.n_segments} steps"
)

## Coarse vs. fine, side by side

A picture of what the budget buys: compare a coarse and a fine point from the
frontier against the original load over one week. The fine one tracks the detail;
the coarse one keeps only the broad shape:

In [ ]:
fine = pareto.find_by_timesteps(84)
week = slice("2010-01-11", "2010-01-17")
comparison = pd.DataFrame(
    {
        "original": data.loc[week, "Load"],
        f"coarse ({small.n_clusters}x{small.n_segments})": small.reconstructed.loc[
            week, "Load"
        ],
        f"fine ({fine.n_clusters}x{fine.n_segments})": fine.reconstructed.loc[
            week, "Load"
        ],
    }
)
px.line(comparison, title="One week: coarse vs. fine aggregation")

Any point on the frontier is a ready-to-use `AggregationResult` — hand it straight
to your model. See the [Optimization workflow](optimization_workflow.ipynb).

## Watch the detail return

Finally, sweep across the whole frontier as an animation. Each frame stacks all
four variables (normalised) as a day-by-hour heatmap of the reconstruction,
starting at the **finest** resolution and compressing from there. As the budget
shrinks, the daily and seasonal structure blurs away. The slider is labelled with
the time-step reduction.

In [ ]:
tuning_result = find_pareto_front(
    data,
    period_duration="1D",
    n_jobs=1,
    numerical_tolerance=1e-8,
    show_progress=False,
)

period_duration = tuning_result.best_result.n_timesteps_per_period  # 24
n_days = len(data) // period_duration
n_vars = len(data.columns)
data_min, data_range = data.min(), data.max() - data.min()

# Each frame: every column's reconstruction, normalised and stacked vertically,
# as a (variables x hours) by (days) heatmap. Start at full resolution (largest
# budget) and compress from there.
frames, labels = [], []
for r in sorted(
    tuning_result.all_results, key=lambda r: r.n_clusters * r.n_segments, reverse=True
):
    reduction = 1 - (r.n_clusters * r.n_segments) / len(data)
    labels.append(f"{reduction:.0%} smaller ({r.n_clusters}d x {r.n_segments}seg)")
    norm = (r.reconstructed - data_min) / data_range
    arr = norm.values.reshape(n_days, period_duration, n_vars).transpose(2, 1, 0)
    frames.append(arr.reshape(-1, n_days))

fig = px.imshow(
    np.stack(frames),
    animation_frame=0,
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    labels={"x": "day", "y": "hour"},
    title="Time series aggregation",
)
for i, step in enumerate(fig.layout.sliders[0].steps):
    step["label"] = labels[i]
ticks = [period_duration * i + period_duration // 2 for i in range(n_vars)]
fig.update_yaxes(tickvals=ticks, ticktext=list(data.columns))
fig.update_layout(height=600, coloraxis_showscale=False)
fig